零方差组是指同一 Prompt 下采样的所有候选回复奖励完全相同，导致组内奖励方差为零、优势值全部归零、无法产生有效梯度更新的样本组。 具体机制与影响如下：
## 产生机制
在 GRPO 等组相对策略优化算法中，模型会对每个 Prompt 生成一组候选回复（通常 G=8~16 条），然后通过组内归一化计算优势值：

A_i = (r_i - μ) / (σ + ε)
- 其中 r_i 是第 i 条回复的奖励，μ 是组内奖励均值，σ 是组内奖励标准差。
- <font color='red'>当组内所有回复的奖励完全相同时（比如全对或全错），标准差 σ = 0，优势值 A_i 全部为零，梯度消失，模型无法从这组样本中学到任何东西。</font>

## 两种典型场景
  - 全成功组：Prompt 太简单，模型所有回复都答对，奖励全是 1。训练后期模型能力提升后尤其常见。
  - 全失败组：Prompt 太难，模型所有回复都答错，奖励全是 0。训练早期尤为突出。
  
## 三大负面影响
  - 梯度为零：<font color='red'>这些样本不产生任何策略更新信号，对模型学习毫无贡献。</font>
  - 算力浪费：生成这些回复消耗了宝贵的推理成本（rollout 是 RL 训练中最贵的环节），但没有产生有效回报。
  - 统计干扰：<font color='red'>在计算 Batch-level 标准差时，零方差样本会人为拉低整体方差，影响归一化效果。</font>
  
## 业界主流应对方案
  - DAPO（动态采样）：
    - 扩大候选 Prompt 池，<font color='red'>过滤掉零方差组，只保留有效组参与训练。缺点是额外 rollout 成本高。</font>
  - POPO（组级回放）：
    - 清华团队提出，不额外 rollout，而是用最近缓存的有效组替换无效组，大幅节省算力。
  - NGRPO（虚拟样本注入）：
    - 向零方差组中注入一个虚拟满分样本，人为制造方差，打破零梯度困境。
  - RL-ZVP（熵引导优势塑形）：
    - 不丢弃零方差组，而是通过熵引导机制从中提取学习信号，实验显示效果甚至优于直接过滤。
  - RLVP（过程奖励补充方差）：
    - 引入可验证的过程惩罚信号，即使结果奖励全相同，过程信号也能恢复组内方差。
    
## 📌 总结
- 零方差组本质上是 RL 训练中的"<font color='red'>无效算力消耗</font>"
  - 模型花了大量计算资源生成回复，但因为缺乏组内差异，无法提供有效的学习信号。它是 RLVR 训练中非常常见的问题，也是当前大模型 RL 后训练效率优化的核心瓶颈之一。